# 01 · Reading is already a quality boundary

A file/read error prevents access to bytes. A parsing error prevents interpretation. A schema problem violates a structural contract. Data validation checks values; business rules relate those values to business meaning. Do not collapse all five into “bad data”.


## Environment
Upload the prepared sample files before the session, then use notebook 00 to check the configured storage. This notebook then runs independently, top to bottom. Spark 3.5 is the target; no Hive catalog is used. Set `BASE_PATH` in the following cell or set `DQ_BASE_PATH` in the driver environment.


In [ ]:
import os
import uuid

from pyspark.sql import SparkSession, Window
from pyspark.sql import functions as F
from pyspark.sql import types as T

spark = SparkSession.builder.appName("OrderDataQuality").getOrCreate()
spark.conf.set("spark.sql.session.timeZone", "UTC")
spark.conf.set("spark.sql.shuffle.partitions", "4")  # tiny teaching datasets only
spark.conf.set("spark.sql.ansi.enabled", "true")
spark.conf.set("spark.sql.csv.parser.columnPruning.enabled", "false")
BASE_PATH = os.environ.get(
    "DQ_BASE_PATH", "s3://YOUR-BUCKET/training/order-quality"
).rstrip("/")
# On EMR/Glue edit the default above if driver environment variables are unavailable.
# Spark VM: hdfs:///user/student/order-quality ; local: file:///tmp/order-quality
assert (
    "YOUR-BUCKET" not in BASE_PATH
), "Set DQ_BASE_PATH or edit BASE_PATH before running"
PROCESSING_DATE = os.environ.get("DQ_PROCESSING_DATE", "2024-01-03")
RUN_ID = uuid.uuid4().hex
RAW_PATH, BRONZE_PATH, SILVER_PATH, GOLD_PATH, QUARANTINE_PATH, AUDIT_PATH = [
    f"{BASE_PATH}/{layer}"
    for layer in ["raw", "bronze", "silver", "gold", "quarantine", "audit"]
]
print("Spark", spark.version, "storage", BASE_PATH, "run", RUN_ID)


## CSV: inspect before interpreting
A is valid, B has a type error, C has fewer fields, D has an extra token, and E parses but has a negative quantity. CSV field-count and quoting behavior varies by Spark parser/version: do not use `_corrupt_record` as a universal CSV shape validator. Disable column pruning for consistent classroom observations.


In [ ]:
csv_path = f"{RAW_PATH}/csv_edge"
spark.read.text(csv_path).show(truncate=False)
csv_schema = "id string, quantity int, price decimal(10,2), _corrupt_record string"
parsed = (
    spark.read.schema(csv_schema)
    .option("header", True)
    .option("mode", "PERMISSIVE")
    .option("columnNameOfCorruptRecord", "_corrupt_record")
    .csv(csv_path)
    .cache()
)
parsed.show(truncate=False)  # materialize full rows before corrupt-column-only queries
parsed.filter("_corrupt_record is null").show(truncate=False)
parsed.filter("_corrupt_record is not null").show(truncate=False)


## Reader modes
PERMISSIVE preserves recoverable rows and corruption evidence when a corrupt field is declared. DROPMALFORMED can lose records before a business check sees them. FAILFAST stops at an action, because Spark is lazy. Do not use a projection-only count to prove that every field was parsed.


In [ ]:
for mode in ["PERMISSIVE", "DROPMALFORMED", "FAILFAST"]:
    print("MODE", mode)
    try:
        spark.read.schema(csv_schema).option("header", True).option("mode", mode).csv(
            csv_path
        ).show(truncate=False)
    except Exception as exc:
        if mode != "FAILFAST":
            raise
        print("Expected parser failure:", type(exc).__name__)


## Inference is observation, not a contract
The value `two` can make an inferred quantity column a string. A later clean file might infer an integer. Explicit schemas stabilize the contract, but you must still detect conversion failures and preserve input. Reader nullability is not a database NOT NULL constraint.


In [ ]:
spark.read.option("header", True).option("inferSchema", True).csv(
    csv_path
).printSchema()
parsed.printSchema()


## JSON: syntax, missing fields, extra fields and wrong types
With an explicit schema, unlisted keys can disappear silently. Missing keys become null; wrong types may produce a corrupt record. Compare the source text as well as the parsed output. Cache the entire parsed DataFrame before selecting only corrupt records.


In [ ]:
json_schema = (
    "id string, quantity int, address struct<city:string>, _corrupt_record string"
)
j = (
    spark.read.schema(json_schema)
    .option("mode", "PERMISSIVE")
    .option("columnNameOfCorruptRecord", "_corrupt_record")
    .json(f"{RAW_PATH}/json_edge")
    .cache()
)
j.show(truncate=False)
j.filter("_corrupt_record is null").show(truncate=False)
j.filter("_corrupt_record is not null").show(truncate=False)
spark.read.json(f"{RAW_PATH}/json_edge").printSchema()


## Production discussion
A source manifest and preserved raw envelopes allow reconciliation even when parsing fails. “Successfully read” does not imply “valid”. XML is omitted to avoid a format-specific dependency. Exercise: replace B’s `two` with `2`, upload the edited file to a separate input prefix, and predict how inferred types change. [Spark CSV reader documentation](https://spark.apache.org/docs/3.5.7/sql-data-sources-csv.html).
